# 06b — Visualizações a partir dos embeddings de teste salvos

Este notebook gera somente as cinco projeções pareadas e o painel. Não lê treino/validação, não inicializa backbones, não treina sonda e não executa KNN/Silhouette. Os vetores completos nunca são materializados: primeiro são lidas apenas as colunas de rótulos/IDs; depois, os grupos de linhas necessários às features selecionadas.

Por padrão, a origem é o run `c01-20261104T161929937502`. Ajuste os parâmetros na primeira célula. Para incluir todos os gêneros elegíveis, use `MAX_GENERA = "all"` e mantenha `MAX_SAMPLES = "auto"`: o orçamento será de dez fragmentos por gênero. Isso não inclui todos os fragmentos do teste. Para milhares de gêneros, o PNG usa seis espaços (cinco modelos e informações), sem legenda sobreposta. O HTML permite buscar/destacar um gênero nos cinco modelos, consultar IDs e usar zoom.

É necessário o de-para taxonômico original, além dos cinco pares `*_test_*.parquet`/`.json`. O notebook procura o de-para registrado no manifesto da origem e a topologia já usada no notebook 06; `TAXONOMY_PATH` permite escolher explicitamente o arquivo correto. `taxonomy_rank_coverage.json` contém apenas cobertura e não substitui o de-para.

Para redesenhar coordenadas existentes sem reler embeddings nem ajustar UMAP, defina `PROJECTION_SOURCE_DIR` com a pasta que contém os cinco CSVs e `umap_comparison.json`. Nesse modo, os parâmetros de amostragem não são aplicados; a amostra salva é preservada.

A verificação SHA-256 continua percorrendo todos os bytes dos **cinco Parquets de teste**, com indicação de tempo. Parquet pode exigir decodificar um grupo inteiro para obter poucas linhas. O ganho é evitar matrizes completas, temporários de spill e todas as etapas de treino/avaliação quantitativa. As projeções são qualitativas e ajustadas independentemente. Consulte `docs/latent_visualizations_only.md`.


In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

# Resolução autônoma para desenvolvimento local e JupyterHub remoto.
current_dir = Path.cwd().resolve()
project_root = None
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        project_root = r
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break
if project_root is None:
    raise RuntimeError("Could not resolve repository root containing src/viral_cnn")

# Parâmetros editáveis; variáveis de ambiente permitem execução sem editar o arquivo.
SOURCE_RUN_ID = os.environ.get("LATENT_VIS_SOURCE_RUN_ID", "c01-20261104T161929937502")
SOURCE_DIR = Path(os.environ.get("LATENT_VIS_SOURCE_DIR", str(
    project_root / "results_reproduction" / "runs" / SOURCE_RUN_ID / "06_ictv_latent_space_and_hierarchy")))
PROJECTION_SOURCE_DIR = os.environ.get("LATENT_VIS_PROJECTION_SOURCE_DIR") or None  # pasta de CSVs/umap_comparison.json já gerados
TAXONOMY_PATH = os.environ.get("TAXONOMY_PATH") or None
MAX_GENERA = os.environ.get("LATENT_VIS_MAX_GENERA", "10")  # "all" para todos os gêneros elegíveis
MAX_SAMPLES = os.environ.get("LATENT_VIS_MAX_SAMPLES", "auto")  # orçamento automático por gênero
SAMPLES_PER_GENUS = int(os.environ.get("LATENT_VIS_SAMPLES_PER_GENUS", "10"))
READ_BATCH_SIZE = int(os.environ.get("LATENT_VIS_READ_BATCH_SIZE", "4096"))
EXECUTION_MODE = os.environ.get("LATENT_VIS_EXECUTION_MODE", "reuse")
REUSE_RUN_ID = os.environ.get("LATENT_VIS_REUSE_RUN_ID") or None
if MAX_GENERA != "all" and int(MAX_GENERA) < 1:
    raise ValueError("MAX_GENERA must be all or a positive integer")
if MAX_SAMPLES != "auto" and int(MAX_SAMPLES) < 3:
    raise ValueError("MAX_SAMPLES must be auto or an integer >= 3")
if READ_BATCH_SIZE < 1 or SAMPLES_PER_GENUS < 1:
    raise ValueError("READ_BATCH_SIZE and SAMPLES_PER_GENUS must be positive")
if EXECUTION_MODE not in ("reuse", "rerun"):
    raise ValueError("EXECUTION_MODE must be reuse or rerun")

import numpy as np
from viral_cnn.notebook_runtime import setup_notebook_runtime, reserve_run_directory, get_git_commit
from viral_cnn.eval.latent import _sha256_file
from viral_cnn.latent_resume import LatentResumeCache, _atomic
from viral_cnn.paired_umap import ProjectionConfig, run_paired_umap, MODEL_NAMES
from viral_cnn.visualization_cache import open_test_embeddings, load_visualization_taxonomy, create_visualization_fixtures
from viral_cnn.latent_plotting import load_saved_comparison, render_comparison
from viral_cnn.utils.logging_utils import log_artifact_path

# Ausência de dados brutos não deve ativar smoke num servidor que tem os caches.
ctx = setup_notebook_runtime(auto_smoke_fallback=False)
run_id = "n06b-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
output_dir = reserve_run_directory(project_root, run_id, "06b_latent_visualizations")
resume = LatentResumeCache(output_dir, mode=EXECUTION_MODE, reuse_run_id=REUSE_RUN_ID)
if ctx.is_smoke_test and PROJECTION_SOURCE_DIR is None:
    SOURCE_DIR, fixture_taxonomy = create_visualization_fixtures(output_dir / "fixtures", seed=ctx.seed)
    TAXONOMY_PATH = str(fixture_taxonomy)
print(f"[VIS_ONLY] Source: {SOURCE_DIR.resolve()}")
print(f"[VIS_ONLY] Output: {output_dir}; smoke={ctx.is_smoke_test}; projection device=CPU")


In [ ]:
# 2. Coordenadas salvas ou embeddings de teste; nunca carregar treino/validação.
if PROJECTION_SOURCE_DIR is not None:
    SOURCE_DIR = Path(PROJECTION_SOURCE_DIR).resolve()
    comparison, projection_inputs = load_saved_comparison(SOURCE_DIR, output_dir)
    embeddings = {}
    tax_path = None
    source_manifest_path = SOURCE_DIR / "manifest.json"
    environment = comparison["metadata"].get("environment", {})
    print("[VIS_ONLY] Apenas redesenho: mesmas coordenadas, sem UMAP, taxonomia ou embeddings.")
else:
    embeddings = open_test_embeddings(SOURCE_DIR, smoke=ctx.is_smoke_test, batch_size=READ_BATCH_SIZE)
    source_run_manifest = {}
    source_manifest_path = SOURCE_DIR / "manifest.json"
    if source_manifest_path.is_file():
        log_artifact_path("source run manifest", source_manifest_path)
        source_run_manifest = json.loads(source_manifest_path.read_text())
    if TAXONOMY_PATH is None:
        # Priorizar a mesma tabela registrada pelo notebook que produziu os embeddings.
        recorded_taxonomies = [Path(p) for p in source_run_manifest.get("input_files", [])
                               if "taxonom" in str(p).lower() and str(p).endswith(".parquet") and Path(p).is_file()]
        TAXONOMY_PATH = str(recorded_taxonomies[0]) if recorded_taxonomies else ctx.topology.paths["short"]["hybrid_taxonomy_parquet"]
    tax_path = Path(TAXONOMY_PATH).resolve()
    if not tax_path.is_file():
        raise FileNotFoundError(f"Taxonomia ausente: {tax_path}. Defina TAXONOMY_PATH com o de-para Parquet real.")
    class_order = embeddings[MODEL_NAMES[0]].manifest["class_order"]
    taxonomy_df, species_to_genus_name = load_visualization_taxonomy(tax_path, class_order)
    base_labels = embeddings[MODEL_NAMES[0]].labels
    eligible_genera = {species_to_genus_name[int(label)] for label in np.unique(base_labels)
                      if int(label) in species_to_genus_name}
    if not eligible_genera:
        raise ValueError("Nenhum gênero com exemplos de teste e mapeamento conhecido")
    genus_limit = len(eligible_genera) if MAX_GENERA == "all" else min(int(MAX_GENERA), len(eligible_genera))
    sample_limit = max(3, SAMPLES_PER_GENUS * genus_limit) if MAX_SAMPLES == "auto" else int(MAX_SAMPLES)
    if MAX_GENERA == "all" and sample_limit < len(eligible_genera):
        raise ValueError("MAX_SAMPLES insuficiente para incluir todos os gêneros; use auto ou um limite maior")
    umap_config = ProjectionConfig(max_samples=sample_limit, max_genera=genus_limit, seed=ctx.seed)
    print(f"[VIS_ONLY] Eligible genera={len(eligible_genera)}; max_genera={genus_limit}; max_samples={sample_limit}")


In [ ]:
# 3. Preservar o ajuste C10 e redesenhar com suporte a milhares de gêneros.
if PROJECTION_SOURCE_DIR is None:
    from importlib.metadata import version, PackageNotFoundError
    optional_versions = {}
    for package in ("umap-learn", "scikit-learn", "scipy", "matplotlib"):
        try:
            optional_versions[package] = version(package)
        except PackageNotFoundError:
            optional_versions[package] = "unavailable"
    environment = {"workflow": "c11_visualization_only_v1", "seed": ctx.seed, "device": "cpu",
                   "smoke": ctx.is_smoke_test, "versions": ctx.library_versions, "optional_versions": optional_versions,
                   "source_files": {name: _sha256_file(project_root / "src/viral_cnn" / name)
                                    for name in ("visualization_cache.py", "paired_umap.py", "latent_resume.py", "eval/latent.py")}}
    display_names = dict(zip(MODEL_NAMES, ("ResNet50", "ResNet50_F", "ConvNeXt_Z", "ConvNeXt_T", "ConvNeXt_F")))
    model_specs = [{"name": name, "display": display_names[name],
                    "test_artifact_sha256": embeddings[name].manifest["artifact_sha256"]} for name in MODEL_NAMES]
    comparison = run_paired_umap(
        model_specs,
        {name: item.features for name, item in embeddings.items()},
        {name: item.labels for name, item in embeddings.items()},
        {name: item.row_ids for name, item in embeddings.items()},
        species_to_genus_name, resume, environment, config=umap_config,
        embedding_inputs={name: {"path": str(item.path), "manifest": item.manifest} for name, item in embeddings.items()},
    )
rendering = render_comparison(comparison, output_dir)
print(f"[VIS_ONLY] {comparison['metadata']['projection_method']}: cinco variantes; "
      f"{len(comparison['sample'].frame)} fragmentos pareados; coordenadas preservadas")
log_artifact_path("visualization panel", output_dir / "umap_comparison.png")
print(f"[VIS_ONLY] Busca por gênero e zoom: {output_dir / 'umap_comparison.html'}")
try:
    from IPython.display import Image, display, FileLink
except ImportError:
    print(f"[VIS_ONLY] Panel: {output_dir / 'umap_comparison.png'}")
else:
    display(Image(filename=str(output_dir / "umap_comparison.png")))
    # Caminho relativo ao cwd permite abrir o HTML pelo Jupyter remoto.
    display(FileLink(os.path.relpath(output_dir / "umap_comparison.html", Path.cwd()),
                     result_html_prefix="Explorar gêneros: "))


In [ ]:
# 4. Manifesto independente: reutilizar hashes já verificados evita reler os Parquets.
inputs = (projection_inputs if PROJECTION_SOURCE_DIR is not None else
          [record for item in embeddings.values() for record in item.inputs])
small_inputs = ([tax_path] if tax_path is not None else []) + ([source_manifest_path] if source_manifest_path.is_file() else [])
evidence_smoke = bool(environment.get("smoke", ctx.is_smoke_test))
inputs += [{"path": str(p.resolve()), "sha256": _sha256_file(p), "size_bytes": p.stat().st_size,
            "quality": "observed"} for p in small_inputs]
outputs = [{"path": str(p.resolve()), "sha256": _sha256_file(p), "size_bytes": p.stat().st_size,
            "quality": "observed"} for p in sorted(output_dir.iterdir()) if p.is_file() and p.name != "manifest.json"]
manifest = {
    "notebook_id": "06b_latent_visualizations", "protocol": "c11_visualization_only_v1",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(), "git_commit": get_git_commit(),
    "run_id": run_id, "source_dir": str(SOURCE_DIR.resolve()), "seed": environment.get("seed", ctx.seed),
    "is_smoke_test": evidence_smoke, "device": "cpu", "library_versions": ctx.library_versions,
    "inputs": inputs, "outputs": outputs, "checkpoints": {},
    "configuration": {"command": "jupyter execute notebooks/reproduction/short/06b_latent_visualizations.ipynb",
                      "max_genera": MAX_GENERA, "max_samples": MAX_SAMPLES,
                      "samples_per_genus": SAMPLES_PER_GENUS, "effective_limits": comparison["sample"].audit["configuration"],
                      "read_batch_size": READ_BATCH_SIZE, "environment": environment,
                      "taxonomy_path": str(tax_path) if tax_path else None,
                      "projection_source_dir": str(SOURCE_DIR) if PROJECTION_SOURCE_DIR is not None else None,
                      "rendering": rendering,
                      "sampling_parameters_applied": PROJECTION_SOURCE_DIR is None, "checkpoint_verification": "archived metadata only; weights not loaded"},
    "read_statistics": {name: item.features.stats for name, item in embeddings.items()},
    "metrics_summary": {"projection_variants": len(comparison["projections"]),
                        "projection_method": comparison["metadata"]["projection_method"],
                        "models_loaded": 0, "train_validation_loaded": False, "quantitative_metrics_computed": False},
    "evidence": {"status": "synthetic" if evidence_smoke else ("observed saved coordinates" if PROJECTION_SOURCE_DIR is not None else "observed archived test embeddings"),
                 "genome_independence": "unknown", "is_qualitative_only": True,
                 "limitation": "No raw data or model checkpoint revalidation; not exact reproduction of published Figure 2."},
}
_atomic(output_dir / "manifest.json", lambda p: p.write_text(json.dumps(manifest, indent=2) + "\n"))
print(json.dumps(manifest["read_statistics"], indent=2))
print(f"[VIS_ONLY] Manifest: {output_dir / 'manifest.json'}")
